# Phase 2 — Generate completions from base + both organisms

See `docs/research_proposal.md` §4.3. Both organisms are compared against the same fixed base-model completions (generated once, reused for both). Grading happens separately, locally, not in Colab -- see `notebooks/grade_local.ipynb`. This notebook only produces `phase2_completions_{A,B}.json`, which you'll download from Drive to run that notebook.

In [ ]:
%pip install -q unsloth peft transformers trl datasets huggingface_hub accelerate bitsandbytes pillow pyyaml anthropic
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_DIR = '/content/drive/MyDrive/emergent-misalignment-project'  # upload src/ here
sys.path.append(PROJECT_DIR)
import os
os.chdir(PROJECT_DIR)  # src/ modules use paths relative to the project root (e.g. data/eval/scenarios.json)
ARTIFACTS = f'{PROJECT_DIR}/artifacts'


## Load base model and eval sets (shared across both organisms)

In [ ]:
from src.train import load_base_model, MODEL_NAME
from src.generate import load_text_eval_prompts, load_multimodal_eval_set

base_model, base_tokenizer = load_base_model(MODEL_NAME)

text_prompts = load_text_eval_prompts()
mm_set = load_multimodal_eval_set()
mm_prompts = [ex['prompt'] for ex in mm_set]
mm_images = [ex['image'] for ex in mm_set]


## Base-model completions (generated once, shared reference for both organisms' judge comparisons)

In [ ]:
import gc

import torch

from src.generate import generate_completions

text_base_completions = generate_completions(
    base_model, base_tokenizer, text_prompts, out_path=f'{ARTIFACTS}/gen_text_base.jsonl',
)
mm_base_completions = generate_completions(
    base_model, base_tokenizer, mm_prompts, images=mm_images, out_path=f'{ARTIFACTS}/gen_mm_base.jsonl',
)

del base_model  # only the completion text is needed downstream, not the model object
gc.collect()
torch.cuda.empty_cache()


## Generate + grade each organism

Writes `phase2_completions_{A,B}.json` and `labels_{text,mm}_{A,B}.jsonl` -- Phase 4 reads these per organism.

In [ ]:
import gc
import json
from pathlib import Path

import torch

for organism in ['A', 'B']:
    checkpoint = Path(f'{ARTIFACTS}/checkpoint_path_{organism}.txt').read_text().strip()
    ft_model, ft_tokenizer = load_base_model(checkpoint)

    text_ft_completions = generate_completions(
        ft_model, ft_tokenizer, text_prompts, out_path=f'{ARTIFACTS}/gen_text_ft_{organism}.jsonl',
    )
    mm_ft_completions = generate_completions(
        ft_model, ft_tokenizer, mm_prompts, images=mm_images, out_path=f'{ARTIFACTS}/gen_mm_ft_{organism}.jsonl',
    )

    del ft_model  # free before the next organism's model loads
    gc.collect()
    torch.cuda.empty_cache()

    Path(f'{ARTIFACTS}/phase2_completions_{organism}.json').write_text(json.dumps({
        'text_prompts': text_prompts,
        'text_base_completions': text_base_completions, 'text_ft_completions': text_ft_completions,
        'mm_prompts': mm_prompts,
        'mm_base_completions': mm_base_completions, 'mm_ft_completions': mm_ft_completions,
    }))

    print(f'organism {organism}: generation done, saved to phase2_completions_{organism}.json')
